In [4]:
import torch
# we perform Tucker Decomposition using tensorly
import tensorly as tl
from tensorly.decomposition import tucker, non_negative_tucker, non_negative_tucker_hals
import time, os, sys, io, re, pickle, inspect
from contextlib import contextmanager
from utils import notify_discord
from utils import DATA_DIR

def tensor_iteration(path_to_tensors=DATA_DIR/"tensors", population_method="counting", top_k=750,
                     n_iter_max=1000,
                     rank = 100,
                     tol = 1e-10,
                     verbose=True,
                     gpu_device=0,
                     init="svd"
):
    # we first check if the work has been done: if so, we skip and return nothing
    decomposition_path = f"{path_to_tensors}/test"
    os.makedirs(decomposition_path, exist_ok=True)
    if os.path.exists(f"{decomposition_path}/{population_method}_{top_k}d_{rank}r_{n_iter_max}i.pkl"):
        print(f"Decomposition for {population_method} with top_k={top_k}, rank={rank} and n_iter_max={n_iter_max} already exists, skipping computation.")
        return None


    os.environ["CUDA_VISIBLE_DEVICES"] = f"{gpu_device}"  #  is being used at time of writing
    tl.set_backend('pytorch')
    torch.cuda.set_device(0)

    # we load in the different tensors
    with open(f"{path_to_tensors}/tensor_{population_method}_{top_k}.pkl", "rb") as f:
        # these were saved as torch tensors!
        try:
            tensor = pickle.load(f)
            # we make sure the tensor is a torch tensor
            if not isinstance(tensor, torch.Tensor):
                tensor = torch.as_tensor(tensor)
        except pickle.UnpicklingError as e:
            # print("Pickle unpickling error, trying torch.load instead.")
            tensor = torch.load(f)
    print(f"Loaded tensor of shape {tensor.shape} from {path_to_tensors}/tensor_{population_method}_{top_k}.pkl")
    print(f"Using {init} initialization for decomposition.")

    # ensure float32 and move to GPU (if they are numpy arrays)
    tensor = torch.as_tensor(tensor, dtype=torch.float32, device='cuda')


    # we load vocabularies associated to the tensor
    # with open(f"{path_to_tensors}/vocabularies_{top_k}.pkl", "rb") as f:
    #     vocab = pickle.load(f)
    # vocab_v = vocab["vocab_v"]
    # vocab_s = vocab["vocab_s"]
    # vocab_o = vocab["vocab_o"]
    # v2i = vocab["v2i"]
    # s2i = vocab["s2i"]
    # o2i = vocab["o2i"]



    def decompose(tensor, non_negative=True, hals=True, rank=None, **kwargs):
        if rank is None:
            rank = [100, 100, 100]
        tic = time.time()

        if not non_negative:
            decomp = tucker
        elif hals:
            decomp = non_negative_tucker_hals
        else:
            decomp = non_negative_tucker


        # figure out what kwargs this function actually accepts
        sig = inspect.signature(decomp)
        allowed_params = sig.parameters.keys()

        # keep only the kwargs that are valid for this specific function
        filtered_kwargs = {k: v for k, v in kwargs.items() if k in allowed_params}


        (core, factors), error = decomp(tensor, rank=rank, **filtered_kwargs)
        toc = time.time()
        print("Time taken for decomposition", toc-tic)
        return (core, factors), error

    # some shenanigans to show iteration numbers live
    _NUM = r"[+-]?(?:\d+(?:\.\d*)?|\.\d+)(?:[eE][+-]?\d+)?"

    class IterPrefixer(io.TextIOBase):
        def __init__(self, orig, on_iter=None, autoflush=True):
            self.orig = orig
            self.buf = ""
            self.iter = 0
            self.on_iter = on_iter
            self.autoflush = autoflush
            # NOTE: no lookahead at the end; we'll strip punctuation when parsing
            self._regex = re.compile(
                rf"reconstruction\s*error\s*=\s*({_NUM})\s*,\s*variation\s*=\s*({_NUM})",
                re.IGNORECASE,
            )

        def write(self, s):
            self.buf += s
            # robust line handling (supports mixed \n / \r\n / partial lines)
            while True:
                nl = self.buf.find("\n")
                if nl == -1:
                    break
                line = self.buf[:nl]
                self.buf = self.buf[nl+1:]

                m = self._regex.search(line)
                if m:
                    self.iter += 1
                    err_str = m.group(1).rstrip(".,;: ")
                    var_str = m.group(2).rstrip(".,;: ")
                    try:
                        err = float(err_str)
                        var = float(var_str)
                    except ValueError:
                        err = var = None  # fail-safe
                    if self.on_iter and (err is not None) and (var is not None):
                        try:
                            self.on_iter(self.iter, err, var)
                        except Exception:
                            pass
                    line = f"[iter {self.iter:03d}] " + line

                self.orig.write(line + "\n")
                if self.autoflush:
                    try: self.orig.flush()
                    except Exception: pass
            return len(s)

        def flush(self):
            if self.buf:
                # write any trailing partial line as-is
                self.orig.write(self.buf)
                self.buf = "\r"
            try: self.orig.flush()
            except Exception: pass

    @contextmanager
    def live_iteration_numbers(on_iter=None):
        old_stdout, old_stderr = sys.stdout, sys.stderr
        pref_out = IterPrefixer(old_stdout, on_iter=on_iter, autoflush=True)
        pref_err = IterPrefixer(old_stderr, on_iter=on_iter, autoflush=True)
        sys.stdout, sys.stderr = pref_out, pref_err
        try:
            yield
        finally:
            sys.stdout, sys.stderr = old_stdout, old_stderr


    with live_iteration_numbers():
        (core, factors), errors_reg = decompose(
            tensor, non_negative=True, hals=False,
            rank=rank, n_iter_max=n_iter_max, init=init, tol=tol, verbose=verbose, return_errors=True
        )
        # we save this to disk
        with open(f"{decomposition_path}/{population_method}_{top_k}d_{rank}r_{n_iter_max}i.pkl", "wb") as f:
            pickle.dump((core, factors), f)
        print(f"Saved decomposition to {decomposition_path}/{population_method}_{top_k}d_{rank}r_{n_iter_max}i.pkl")
        print(f"Final reconstruction error: {errors_reg[-1]}")
    return (core, factors), errors_reg

In [ ]:
method = "sc"
path = DATA_DIR/"tensors/fineweb_dutch_vectors_ids"
rank = 100
top_k = 1000

(core, factors), errors_reg = tensor_iteration(path, method, init="random", rank=rank, top_k=top_k, n_iter_max=100, tol=1e-100, gpu_device=1)

Loaded tensor of shape torch.Size([1000, 1000, 1000]) from data/tensors/fineweb_dutch_vectors_ids/tensor_sc_1000.pkl
Using random initialization for decomposition.
[iter 001] reconstruction error=1.0, variation=0.0.
[iter 002] reconstruction error=1.0, variation=0.0.
[iter 003] reconstruction error=1.0, variation=0.0.
[iter 004] reconstruction error=1.0, variation=0.0.
[iter 005] reconstruction error=1.0, variation=0.0.
[iter 006] reconstruction error=1.0, variation=0.0.
[iter 007] reconstruction error=1.0, variation=0.0.
[iter 008] reconstruction error=1.0, variation=0.0.
[iter 009] reconstruction error=1.0, variation=0.0.
[iter 010] reconstruction error=1.0, variation=0.0.
[iter 011] reconstruction error=1.0, variation=0.0.
[iter 012] reconstruction error=1.0, variation=0.0.
[iter 013] reconstruction error=1.0, variation=0.0.
[iter 014] reconstruction error=1.0, variation=0.0.
[iter 015] reconstruction error=1.0, variation=0.0.
[iter 016] reconstruction error=1.0, variation=0.0.
[ite

In [7]:
errors_reg

[tensor(1., device='cuda:0'),
 tensor(1., device='cuda:0'),
 tensor(1., device='cuda:0')]

In [13]:
# we load in the sc 1000d tensor for inspection
import pickle
with open(DATA_DIR/"tensors/fineweb_dutch_vectors_ids/tensor_sii_1000.pkl", "rb") as f:
    tensor = torch.load(f)
tensor.shape

torch.Size([1000, 1000, 1000])

In [14]:
# we check for positive, negative and non-zero entries
print("Number of positive entries:", (tensor > 0).sum().item())
print("Number of negative entries:", (tensor < 0).sum().item())
print("Number of non-zero entries:", (tensor != 0).sum().item())
print("Number of zero entries:", (tensor == 0).sum().item())

Number of positive entries: 176746
Number of negative entries: 45875
Number of non-zero entries: 222621
Number of zero entries: 999777379


In [16]:
with open(DATA_DIR/"tensors/fineweb_dutch_vectors_ids/tensor_sc_1000.pkl", "rb") as f:
    tensor = torch.load(f)
tensor.shape
# we check for positive, negative and non-zero entries
print("Number of positive entries:", (tensor > 0).sum().item())
print("Number of negative entries:", (tensor < 0).sum().item())
print("Number of non-zero entries:", (tensor != 0).sum().item())
print("Number of zero entries:", (tensor == 0).sum().item())

Number of positive entries: 20950
Number of negative entries: 201671
Number of non-zero entries: 222621
Number of zero entries: 999777379
